# TASK 1

## Part 1 - Data Preprocessing

## Importing the libraries

In [1]:
import numpy as np
import pandas as pd

## Importing the dataset

In [2]:
dataset = pd.read_csv('Churn_Modelling.csv')

## Encoding categorical data

## Label Encoding the "Gender" column

In [3]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
label_encoder = LabelEncoder()
dataset['Gender'] = label_encoder.fit_transform(dataset['Gender'])

## One Hot Encoding the "Geography" column

In [4]:
dataset = pd.get_dummies(dataset, columns=['Geography'], drop_first=True)

## Splitting the dataset into the Training set and Test set

In [5]:
from sklearn.model_selection import train_test_split
X = dataset.drop(['Surname', 'Exited'], axis=1) 
y = dataset['Exited']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

## Feature Scaling

In [6]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Part 2 - Building the ANN

## Initializing the ANN and setting its parameters

In [7]:
def initialize_parameters(input_size):
    np.random.seed(0)
    input_layer_size = input_size
    hidden_layer_size = 50
    output_layer_size = 1
    
    weights_input_hidden = np.random.randn(input_layer_size, hidden_layer_size)
    bias_hidden = np.zeros((1, hidden_layer_size))
    
    weights_hidden_output = np.random.randn(hidden_layer_size, output_layer_size)
    bias_output = np.zeros((1, output_layer_size))
    
    return weights_input_hidden, bias_hidden, weights_hidden_output, bias_output

In [8]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [9]:
def forward_propagation(X, weights_input_hidden, bias_hidden, weights_hidden_output, bias_output):
    input_hidden = np.dot(X, weights_input_hidden) + bias_hidden
    output_hidden = sigmoid(input_hidden)
    
    input_output = np.dot(output_hidden, weights_hidden_output) + bias_output
    output_network = sigmoid(input_output)
    
    return output_network

In [10]:
def train_neural_network(X_train, y_train, num_iterations, learning_rate):
    input_size = X_train.shape[1]
    weights_input_hidden, bias_hidden, weights_hidden_output, bias_output = initialize_parameters(input_size)
    
    for epoch in range(num_iterations):
        # Forward propagation
        input_hidden = np.dot(X_train, weights_input_hidden) + bias_hidden
        output_hidden = sigmoid(input_hidden)
        
        output_network = forward_propagation(X_train, weights_input_hidden, bias_hidden, weights_hidden_output, bias_output)
        
        # Backpropagation
        d_output = output_network - y_train.values.reshape(-1, 1)
        d_hidden = d_output.dot(weights_hidden_output.T) * (output_hidden * (1 - output_hidden))
        
        # Update weights and biases
        weights_hidden_output -= output_hidden.T.dot(d_output) * learning_rate
        bias_output -= np.sum(d_output, axis=0, keepdims=True) * learning_rate
        weights_input_hidden -= X_train.T.dot(d_hidden) * learning_rate
        bias_hidden -= np.sum(d_hidden, axis=0, keepdims=True) * learning_rate
    
    return weights_input_hidden, bias_hidden, weights_hidden_output, bias_output

## Training the neural network

In [11]:
num_iterations = 1000
learning_rate = 0.01
weights_input_hidden, bias_hidden, weights_hidden_output, bias_output = train_neural_network(X_train, y_train, num_iterations, learning_rate)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_3508\1253679003.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


## Making predictions on the test set

In [12]:
output_network_test = forward_propagation(X_test, weights_input_hidden, bias_hidden, weights_hidden_output, bias_output)

In [13]:
threshold = 0.5
y_pred = (output_network_test > threshold).astype(int)

In [14]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("Accuracy:", accuracy)

Confusion Matrix:
[[1595    0]
 [ 405    0]]
Accuracy: 0.7975


# TASK 2

## Import Libraries and Initialize the CNN

In [32]:
import numpy as np
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.preprocessing.image import ImageDataGenerator
from keras.preprocessing import image

cnn = Sequential()

## Add Convolutional and Pooling Layers

In [33]:
cnn.add(Conv2D(32, (3, 3), input_shape=(64, 64, 3), activation='relu'))

cnn.add(MaxPooling2D(pool_size=(2, 2)))

# Adding a second convolutional layer
cnn.add(Conv2D(64, (3, 3), activation='relu'))
cnn.add(MaxPooling2D(pool_size=(2, 2)))

## Flattening and Full Connection Layers

In [34]:
cnn.add(Flatten())

cnn.add(Dense(units=128, activation='relu'))
cnn.add(Dense(units=1, activation='sigmoid'))

## Compile the CNN Model

In [35]:
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## Preprocess the Images Using Data Generators

In [36]:
train_datagen = ImageDataGenerator(rescale=1./255, shear_range=0.2, zoom_range=0.2, horizontal_flip=True)
test_datagen = ImageDataGenerator(rescale=1./255)

training_set = train_datagen.flow_from_directory('dataset/dataset/training_set', target_size=(64, 64), batch_size=32, class_mode='binary')
test_set = test_datagen.flow_from_directory('dataset/dataset/test_set', target_size=(64, 64), batch_size=32, class_mode='binary')

Found 8000 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.


## Train the CNN Model

In [37]:
cnn.fit(training_set, steps_per_epoch=8000, epochs=25, validation_data=test_set, validation_steps=2000)

Epoch 1/25
8000/8000 [==============================] - 40s 5ms/step - loss: 0.6758 - accuracy: 0.5794 - val_loss: 0.5993 - val_accuracy: 0.6790


## Make a Single Prediction

In [38]:
test_image = image.load_img('dataset/dataset/single_prediction/cat_or_dog_1.jpg', target_size=(64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis=0)
result = cnn.predict(test_image)

if result[0][0] == 1:
    prediction = 'dog'
else:
    prediction = 'cat'
print(prediction)

1/1 [==============================] - 0s 60ms/step
dog
